<a href="https://colab.research.google.com/github/tanujkumarpandranki/Brain_Tumor_DL_Project/blob/main/Brain_Tumor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Libraries for deep learning, image preprocessing, model building, and training optimization
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [ ]:
# Data augmentation and normalization for improving model generalization
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.3,
    horizontal_flip=True,
    shear_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1
)

In [ ]:
# Load and preprocess training images from directory with resizing and batching
train_data = train_gen.flow_from_directory(
   "/content/drive/MyDrive/BRAIN TUMOR/brain_tumor_dataset/brain_tumor_classification/Training",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical"
)

# Load and preprocess validation/testing images for model evaluation
val_data = train_gen.flow_from_directory(
   "/content/drive/MyDrive/BRAIN TUMOR/brain_tumor_dataset/brain_tumor_classification/Training",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical"
)

Found 2870 images belonging to 4 classes.
Found 2870 images belonging to 4 classes.


In [ ]:
# Load MobileNetV2
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

In [ ]:
# Unfreezing last 30 layers for fine-tuning the pre-trained model
for layer in base_model.layers[-30:]:
    layer.trainable = True

In [ ]:
# Build custom classifier layers for brain tumor classification
x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(128, activation='relu')(x)
x = Dropout(0.6)(x)   # increase dropout

predictions = Dense(4, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

In [ ]:
# Callbacks to stop early and reduce learning rate when validation performance stops improving
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=2,
    min_lr=1e-6
)

In [ ]:
# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,422,468 (9.24 MB)

 Trainable params: 2,388,356 (9.11 MB)

 Non-trainable params: 34,112 (133.25 KB)

In [ ]:
# Train the deep learning model using training data and validate performance on validation data
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/15
90/90 ━━━━━━━━━━━━━━━━━━━━ 459s 4s/step - accuracy: 0.6568 - loss: 0.8611 - val_accuracy: 0.3763 - val_loss: 2.4563 - learning_rate: 1.0000e-04
Epoch 2/15
90/90 ━━━━━━━━━━━━━━━━━━━━ 93s 1s/step - accuracy: 0.8422 - loss: 0.4380 - val_accuracy: 0.4307 - val_loss: 2.5708 - learning_rate: 1.0000e-04
Epoch 3/15
90/90 ━━━━━━━━━━━━━━━━━━━━ 94s 1s/step - accuracy: 0.8941 - loss: 0.2989 - val_accuracy: 0.4652 - val_loss: 2.3832 - learning_rate: 1.0000e-04
Epoch 4/15
90/90 ━━━━━━━━━━━━━━━━━━━━ 94s 1s/step - accuracy: 0.9118 - loss: 0.2476 - val_accuracy: 0.5575 - val_loss: 1.7934 - learning_rate: 1.0000e-04
Epoch 5/15
90/90 ━━━━━━━━━━━━━━━━━━━━ 94s 1s/step - accuracy: 0.9341 - loss: 0.1844 - val_accuracy: 0.6662 - val_loss: 1.2106 - learning_rate: 1.0000e-04
Epoch 6/15
90/90 ━━━━━━━━━━━━━━━━━━━━ 95s 1s/step - accuracy: 0.9408 - loss: 0.1818 - val_accuracy: 0.7017 - val_loss: 1.1249 - learning_rate: 1.0000e-04
Epoch 7/15
90/90 ━━━━━━━━━━━━━━━━━━━━ 92s 1s/step - accuracy: 0.9523 - loss

In [ ]:
# Save trained model and download it from Google Colab
model.save("/content/brain_tumor_model.h5")

from google.colab import files
files.download("/content/brain_tumor_model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Save model to Drive
model.save('/content/drive/MyDrive/brain_tumor_model.h5')

print("✅ Model saved to Google Drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Model saved to Google Drive


In [ ]:
# Display class labels mapping for training and validation datasets
print("Train:", train_data.class_indices)
print("Val:", val_data.class_indices)

{'glioma_tumor': 0, 'meningioma_tumor': 1, 'no_tumor': 2, 'pituitary_tumor': 3}
Val: {'glioma_tumor': 0, 'meningioma_tumor': 1, 'no_tumor': 2, 'pituitary_tumor': 3}


In [ ]:
classes = list(train_data.class_indices.keys())
print(classes)

['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']


In [ ]:

# Upload and preprocess MRI image to predict tumor type using the trained model
from google.colab import files
from PIL import Image
import numpy as np

# upload image
uploaded = files.upload()
image_path = list(uploaded.keys())[0]

# open image
img = Image.open(image_path)
img = img.convert("RGB")
img = img.resize((224,224))

img = np.array(img)/255.0
img = np.expand_dims(img, axis=0)

# predict
prediction = model.predict(img)

index = np.argmax(prediction)
result = classes[index]

print("Prediction:", result)

# Display diagnosis information based on the predicted brain tumor type
if result == "glioma_tumor":
    print("Diagnosis: Glioma Tumor Detected. Please consult a neurologist.")

elif result == "meningioma_tumor":
    print("Diagnosis: Meningioma Tumor Detected. Please consult a doctor for further evaluation.")

elif result == "no_tumor":
    print("Diagnosis: No Brain Tumor Detected. The MRI scan appears normal.")

elif result == "pituitary_tumor":
    print("Diagnosis: Pituitary Tumor Detected. Medical consultation is recommended.")



Saving gg (74).jpg to gg (74).jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Prediction: glioma_tumor
Diagnosis: Glioma Tumor Detected. Please consult a neurologist.
